# Requirements Agent: RAG Pipeline with Hybrid Search and Reranking

This notebook demonstrates a Retrieval Augmented Generation (RAG) pipeline designed to efficiently search and retrieve relevant safety requirements from a given dataset. It leverages a hybrid search approach combining lexical (BM25S) and dense (FAISS with Sentence Transformers) retrieval methods, followed by a Cross-Encoder for reranking to improve the relevance of results.

## 1. Setup and Installation

This section installs the necessary Python libraries for building the RAG pipeline. Key libraries include:

*   `faiss-cpu`: For efficient similarity search on dense vectors.
*   `sentence-transformers`: For generating dense vector embeddings from text.
*   `bm25s`: For lexical keyword-based search.
*   `flashrank`: For the Cross-Encoder reranking model.

```python
pip install faiss-cpu sentence-transformers bm25s flashrank
```

## 2. Persistence Configuration and Data Loading

This section defines the configuration for persisting generated embeddings and parsed data to Google Drive, ensuring that the RAG pipeline can be quickly reloaded across different Colab sessions. It also includes the `load_requirements_dataset` function responsible for parsing raw JSONL data into a structured format suitable for indexing.

### Key Components:

*   **`SAVE_DIR`**: Specifies the Google Drive directory for caching. This helps avoid re-computation of time-consuming steps like embedding generation.
*   **`load_requirements_dataset(file_path)`**:
    *   Checks for cached `documents.json` and `metadata.json` files on Google Drive.
    *   If no cache is found, it reads a raw JSONL file (e.g., `iso26262_train.jsonl`).
    *   Parses each item to extract user messages (requirements) and assistant analyses (ASIL, reasoning).
    *   Constructs a `searchable_content` string for vector indexing.
    *   Stores original requirement text, ASIL, and reasoning in `metadata_storage`.
    *   Saves the parsed `documents` and `metadata_storage` to Google Drive for future use.


## 3. Vector Indexing Generation and Loading

This section handles the creation and loading of the FAISS vector index, which is crucial for efficient dense vector similarity search. It utilizes a Sentence Transformer model to convert text documents into numerical embeddings.

### Key Components:

*   **`build_vector_database(documents)`**:
    *   Initializes a `SentenceTransformer` model (e.g., `Qwen/Qwen3-Embedding-0.6B`) on the available device (GPU if CUDA is available, otherwise CPU).
    *   Leverages `torch.float16` and `attn_implementation='sdpa'` for reduced memory footprint on GPUs.
    *   **Caching**: Checks if a pre-built FAISS index (`faiss_index.bin`) exists on Google Drive. If so, it loads the existing index.
    *   **Embedding Generation**: If no cached index is found, it encodes the `documents` into dense vector embeddings.
    *   **FAISS Index Creation**: Creates an `IndexFlatL2` FAISS index and adds the generated embeddings.
    *   **Persistence**: Saves the newly created FAISS index to Google Drive.


## 4. Database Initialization and Model Loading

This cell orchestrates the loading of all necessary data and models, making the RAG pipeline ready for querying. It performs the following steps:

1.  **Load Documents and Metadata**: Calls `load_requirements_dataset` to get the processed text documents and their associated metadata.
2.  **Build/Load FAISS Index**: Calls `build_vector_database` to initialize the dense vector index and the embedding model.
3.  **Build BM25S Lexical Index**: Creates a `bm25s` retriever for sparse keyword-based search. This index is built quickly from the `documents` and does not require caching.
4.  **Initialize Cross-Encoder Reranker**: Loads a `CrossEncoder` model (e.g., `cross-encoder/ms-marco-MiniLM-L-6-v2`) which is used in the final stage to re-rank the candidate documents based on their semantic relevance to the query.

This section ensures all components of the hybrid retrieval system are initialized and ready.

## 5. RAG Pipeline: Hybrid Search and Reranking

This section defines the core `hybrid_search_and_rerank` function, which implements a two-stage retrieval process:

### Stage 1: Candidate Pool Generation (Hybrid Retrieval with RRF)

1.  **Query Enrichment**: The `enrich_query` function expands the original query with domain-specific synonyms to improve retrieval recall for technical terms.
2.  **BM25S Retrieval**: Performs a lexical search using the `bm25_retriever` to find documents with keyword overlap.
3.  **FAISS Retrieval**: Performs a dense vector similarity search using the `embedding_model` and `faiss_index` to find semantically similar documents.
4.  **Reciprocal Rank Fusion (RRF)**: Combines the results from both BM25S and FAISS using RRF, which weights higher-ranked results from each retriever more heavily. This generates a diverse candidate pool (`pool_size`) of potentially relevant documents.

### Stage 2: Reranking with Cross-Encoder

1.  **Pair Formation**: Creates query-document pairs from the RRF-generated candidate pool.
2.  **Cross-Encoder Prediction**: The `reranker_model` scores each query-document pair based on their semantic similarity. Cross-encoders are generally more accurate than bi-encoders for relevance scoring.
3.  **Final Ranking**: Sorts the candidates by their Cross-Encoder score to present the most relevant documents as the final output (`k`).

This two-stage approach balances the strengths of lexical and semantic search, providing a robust and accurate retrieval system.

## 6. Interactive Stress Testing Block

This section provides a demonstration of the RAG pipeline using a set of `experiment_queries`. It iterates through these queries, applies the `hybrid_search_and_rerank` function, and prints the top matching requirements along with their metadata and reranking scores.

This block allows for quick evaluation of the RAG pipeline's performance on various types of queries, highlighting its ability to retrieve and rank relevant safety requirements based on their content, ASIL rating, and reasoning.

In [ ]:
pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 102.4 MB/s eta 0:00:00


In [ ]:
pip install bm25s flashrank

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.3 MB/s eta 0:00:00


In [ ]:
import json
import os
import sys
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import bm25s

# ---------------------------------------------------------------------------
# 0. PERSISTENCE CONFIGURATION
# ---------------------------------------------------------------------------
# This directory lives on your Google Drive and survives runtime resets!
SAVE_DIR = "/content/drive/MyDrive/Requirements Agent/colab_embeddings_cache"
FAISS_INDEX_PATH = os.path.join(SAVE_DIR, "faiss_index.bin")
DOCUMENTS_PATH = os.path.join(SAVE_DIR, "documents.json")
METADATA_PATH = os.path.join(SAVE_DIR, "metadata.json")

# ---------------------------------------------------------------------------
# 1. LOAD AND PARSE DATASET (With Caching)
# ---------------------------------------------------------------------------
def load_requirements_dataset(file_path):
    # Check if we already have the processed docs and metadata saved
    if os.path.exists(DOCUMENTS_PATH) and os.path.exists(METADATA_PATH):
        print("📂 Loading cached documents and metadata from Google Drive...")
        with open(DOCUMENTS_PATH, "r", encoding="utf-8") as f:
            documents = json.load(f)
        with open(METADATA_PATH, "r", encoding="utf-8") as f:
            metadata_storage = json.load(f)
        print(f"✅ Loaded {len(documents)} documents from cache.")
        return documents, metadata_storage

    documents = []
    metadata_storage = []
    skipped_count = 0

    if not os.path.exists(file_path):
        print(f"❌ Error: Raw dataset file '{file_path}' not found.")
        sys.exit(1)

    print(f"📂 Loading and parsing raw dataset from {file_path}...")

    with open(file_path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)
        if first_char == '[':
            raw_data = json.load(f)
        else:
            raw_data = [json.loads(line) for line in f if line.strip()]

    for idx, item in enumerate(raw_data):
        try:
            user_msg = next(m["content"] for m in item["messages"] if m["role"] == "user")
            assist_msg = next(m["content"] for m in item["messages"] if m["role"] == "assistant")

            req_text = user_msg.split("Analyze this safety requirement:")[-1].strip()

            try:
                analysis = json.loads(assist_msg)
                asil = analysis.get("asil", "QM")
                reasoning = analysis.get("reasoning", "")
            except json.JSONDecodeError:
                asil = "Unknown"
                reasoning = assist_msg

            searchable_content = f"ID: REQ-GEN-{idx:04d} Type: {asil} Requirement: {req_text} Safety Reasoning: {reasoning}"
            documents.append(searchable_content)

            metadata_storage.append({
                "id": f"REQ-GEN-{idx:04d}",
                "req_text": req_text,
                "asil": asil,
                "reasoning": reasoning
            })
        except Exception as e:
            skipped_count += 1
            continue

    print(f"✅ Parsed {len(documents)} requirements (Skipped: {skipped_count}). Saving to Google Drive...")

    # Save parsed data so we don't have to parse raw files again
    os.makedirs(SAVE_DIR, exist_ok=True)
    with open(DOCUMENTS_PATH, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)
    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(metadata_storage, f, ensure_ascii=False, indent=2)

    return documents, metadata_storage


# ---------------------------------------------------------------------------
# 2. VECTOR INDEXING GENERATION & LOADING
# ---------------------------------------------------------------------------
def build_vector_database(documents):
    model_name = "Qwen/Qwen3-Embedding-0.6B"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Clean GPU memory before starting to avoid fragmentation OOM issues
    if device == "cuda":
        print("🧹 Clearing CUDA memory cache...")
        torch.cuda.empty_cache()

    print(f"\n🧠 Initializing Qwen3 Model ('{model_name}') on {device.upper()}...")

    # Load SentenceTransformer model in float16 to save ~8GB of VRAM
    embedding_model = SentenceTransformer(
        model_name,
        trust_remote_code=True,
        device=device,
        model_kwargs={"torch_dtype": torch.float16, "attn_implementation": "sdpa"}
    )

    # Check if FAISS index is already saved on Drive
    if os.path.exists(FAISS_INDEX_PATH):
        print("💾 Found existing FAISS index on Google Drive! Loading...")
        index = faiss.read_index(FAISS_INDEX_PATH)
        print(f"✅ FAISS index loaded successfully. Total vectors: {index.ntotal}")
        return embedding_model, index

    print("⏳ Cache not found. Generating embeddings (this may take a while on a free GPU)...")

    # Lower the batch_size to prevent memory spikes during inference
    embeddings = embedding_model.encode(
        documents,
        batch_size=16,
        show_progress_bar=True
    )
    embeddings = np.array(embeddings).astype("float32")

    dimension = embeddings.shape[1]
    print(f"📐 Detected Embedding Dimensions: {dimension}")

    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    # Save FAISS Index to Google Drive instantly
    print("💾 Saving generated FAISS index to Google Drive...")
    os.makedirs(SAVE_DIR, exist_ok=True)
    faiss.write_index(index, FAISS_INDEX_PATH)

    print(f"📦 FAISS index populated and saved with {index.ntotal} vectors.")
    return embedding_model, index

# ---------------------------------------------------------------------------
# 3. BUILD DATABASES
# ---------------------------------------------------------------------------
DATA_FILE = "/content/iso26262_train.jsonl"

# 1. Parse and build/load pipelines
docs, metadata = load_requirements_dataset(DATA_FILE)

# 2. Build or load FAISS vector database
model, faiss_index = build_vector_database(docs)

# 3. Build bm25s keyword search indexing (very fast to build, no need to cache)
print("\n📝 Initializing optimized bm25s keyword search indexing...")
corpus_tokens = bm25s.tokenize(docs)
bm25_retriever = bm25s.BM25(method="lucene", k1=1.5, b=0.75)
bm25_retriever.index(corpus_tokens)
print("⚡ bm25s sparse index built successfully.")

# 4. Initialize Cross-Encoder Reranker
print("\n🎯 Initializing Cross-Encoder Reranker ('ms-marco-MiniLM-L-6-v2')...")
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("⚡ Reranker loaded and ready.")


# ---------------------------------------------------------------------------
# 4. HYBRID SEARCH & RERANKING PIPELINE
# ---------------------------------------------------------------------------
def retrieve_and_rerank(query, top_k_final=4, pool_size=25, rrf_constant=60):
    """
    Retrieves, fuses (RRF), and reranks documents.
    """
    # --- Step 4.1: BM25 Retrieval ---
    query_tokens = bm25s.tokenize(query)
    bm25_docs, bm25_scores = bm25_retriever.retrieve(query_tokens, k=pool_size)
    bm25_results = []
    for doc in bm25_docs[0]:
        doc_idx = docs.index(doc)
        bm25_results.append(doc_idx)

    # --- Step 4.2: Vector Retrieval ---
    query_vector = model.encode([query]).astype("float32")
    _, faiss_indices = faiss_index.search(query_vector, pool_size)
    faiss_results = list(faiss_indices[0])

    # --- Step 4.3: Reciprocal Rank Fusion (RRF) ---
    rrf_scores = {}

    for rank, doc_idx in enumerate(bm25_results):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (rrf_constant + rank + 1))

    for rank, doc_idx in enumerate(faiss_results):
        if doc_idx != -1:
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (rrf_constant + rank + 1))

    sorted_candidates = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:pool_size]

    # --- Step 4.4: Cross-Encoder Reranking ---
    pairs = [(query, docs[idx]) for idx in sorted_candidates]
    rerank_scores = reranker_model.predict(pairs)

    reranked_results = []
    for score, doc_idx in zip(rerank_scores, sorted_candidates):
        meta = metadata[doc_idx]
        reranked_results.append({
            "id": meta["id"],
            "req_text": meta["req_text"],
            "asil": meta["asil"],
            "reasoning": meta["reasoning"],
            "rerank_score": float(score)
        })

    reranked_results = sorted(reranked_results, key=lambda x: x["rerank_score"], reverse=True)

    return reranked_results[:top_k_final]

📂 Loading cached documents and metadata from Google Drive...
✅ Loaded 859 documents from cache.
🧹 Clearing CUDA memory cache...

🧠 Initializing Qwen3 Model ('Qwen/Qwen3-Embedding-0.6B') on CUDA...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

⚡ Reranker loaded and ready.


In [ ]:
import os
import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import bm25s
from collections import Counter

# ---------------------------------------------------------------------------
# 0. RESOLVE VARIABLES & LOAD CACHE FROM GOOGLE DRIVE IF MISSING
# ---------------------------------------------------------------------------
SAVE_DIR = "/content/drive/MyDrive/Requirements Agent/colab_embeddings_cache"
FAISS_INDEX_PATH = os.path.join(SAVE_DIR, "faiss_index.bin")
DOCUMENTS_PATH = os.path.join(SAVE_DIR, "documents.json")
METADATA_PATH = os.path.join(SAVE_DIR, "metadata.json")

# Check if we need to load variables from Drive (makes Cell 2 fully independent)
if 'docs' not in globals() or 'metadata' not in globals() or 'faiss_index' not in globals():
    print("🔄 Initializing environment and loading saved RAG embeddings from Google Drive...")

    # 1. Load documents and metadata
    if not os.path.exists(DOCUMENTS_PATH) or not os.path.exists(METADATA_PATH):
        raise FileNotFoundError(f"❌ Cache not found in {SAVE_DIR}. Please run Cell 1 first to generate your vector database!")

    print("📂 Loading cached documents and metadata...")
    with open(DOCUMENTS_PATH, "r", encoding="utf-8") as f:
        docs = json.load(f)
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print(f"✅ Loaded {len(docs)} documents.")

    # 2. Load FAISS Index
    print("💾 Loading FAISS Vector Index...")
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    print(f"✅ FAISS Index loaded. Dimensions: {faiss_index.d}, Total Vectors: {faiss_index.ntotal}")

    # 3. Load Embedding Model (Ensuring FP16)
    model_name = "Qwen/Qwen3-Embedding-0.6B"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"🧠 Loading {model_name} in FP16 on {device.upper()}...")
    model = SentenceTransformer(
        model_name,
        trust_remote_code=True,
        device=device,
        model_kwargs={"torch_dtype": torch.float16, "attn_implementation": "sdpa"}
    )

    # 4. Rebuild fast BM25S Lexical Index (takes < 1 second, no need to cache)
    print("📝 Building BM25S sparse index...")
    corpus_tokens = bm25s.tokenize(docs)
    bm25_retriever = bm25s.BM25(method="lucene", k1=1.5, b=0.75)
    bm25_retriever.index(corpus_tokens)
    print("⚡ BM25S sparse index built.")

    # 5. Load Cross-Encoder Reranker
    print("🎯 Loading Cross-Encoder Reranker...")
    reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    print("🚀 System fully initialized from cache!")
else:
    print("⚡ Variables already active in memory! Skipping Google Drive reload to save time.")


# ---------------------------------------------------------------------------
# 1. AUTOMOTIVE SYNONYM EXPANSION MAP
# ---------------------------------------------------------------------------
ENGINEERING_SYNONYM_MAP = {
    "operating temperature": ["temperature", "°C", "ambient", "thermal", "overheating", "cold-start"],
    "environmental conditions": ["temperature", "humidity", "weather", "operating environment"],
    "v2x": ["communication", "signal", "transmission", "highway", "message", "wireless"],
    "communication constraints": ["v2x", "signal", "transmission", "bus", "can fd", "ethernet"],
    "dms": ["distraction", "driver monitoring", "camera", "gaze", "tracking"],
    "driver distraction": ["dms", "monitoring", "alert", "attention"]
}

def enrich_query(query: str) -> str:
    """Appends technical domain terms to help dense/lexical layers catch matches."""
    lowered_query = query.lower()
    expansions = []
    for key, values in ENGINEERING_SYNONYM_MAP.items():
        if key in lowered_query:
            expansions.extend(values)

    if expansions:
        enriched = query + " " + " ".join(list(set(expansions)))
        return enriched
    return query


# ---------------------------------------------------------------------------
# 2. TWO-STAGE HYBRID RETRIEVAL & RERANKING
# ---------------------------------------------------------------------------
def hybrid_search_and_rerank(query, embedding_model, index, bm25_retriever, reranker, metadata_storage, documents, k=3, pool_size=25, rrf_constant=60):
    """
    Executes a two-stage retrieval pipeline:
    Stage 1: Generates a wide candidate pool using RRF (BM25S + FAISS).
    Stage 2: Reranks the candidate pool using a Cross-Encoder to prioritize semantic matching.
    """
    num_docs = len(documents)
    actual_pool_size = min(pool_size, num_docs)

    # --- Step A: Query Enrichment ---
    processed_query = enrich_query(query)

    # --- Step B: Lexical BM25S Scoring ---
    query_tokens = bm25s.tokenize(processed_query)
    bm25_results = bm25_retriever.retrieve(query_tokens, k=actual_pool_size)
    bm25_ranked_docs = bm25_results.documents[0]

    # --- Step C: Dense FAISS Scoring ---
    query_vector = embedding_model.encode([processed_query]).astype("float32")
    _, faiss_ranked_indices = index.search(query_vector, actual_pool_size)
    faiss_ranked_indices = faiss_ranked_indices[0]

    # --- Step D: Reciprocal Rank Fusion (Stage 1 Pool Generation) ---
    rrf_scores = Counter()

    # Score Lexical Positions (find index of document text in corpus)
    for rank, doc_text in enumerate(bm25_ranked_docs):
        try:
            idx = documents.index(doc_text)
            rrf_scores[int(idx)] += 1.0 / (rrf_constant + rank + 1)
        except ValueError:
            continue

    # Score Vector Positions
    for rank, idx in enumerate(faiss_ranked_indices):
        if idx != -1:
            rrf_scores[int(idx)] += 1.0 / (rrf_constant + rank + 1)

    # Grab candidates for Stage 2 reranking
    candidate_hits = rrf_scores.most_common(actual_pool_size)
    candidate_indices = [doc_idx for doc_idx, _ in candidate_hits]

    if not candidate_indices:
        return []

    # --- Step E: Cross-Encoder Reranking (Stage 2 Filtering) ---
    pairs = [(query, documents[idx]) for idx in candidate_indices]
    rerank_scores = reranker.predict(pairs)

    # Map scores back to metadata
    reranked_results = []
    for score, doc_idx in zip(rerank_scores, candidate_indices):
        rrf_score = rrf_scores[doc_idx]

        reranked_results.append({
            "metadata": metadata_storage[doc_idx],
            "rerank_score": float(score),
            "rrf_score": rrf_score,
            "raw_block": documents[doc_idx]
        })

    # Sort candidates by Cross-Encoder score (highest relevance first)
    reranked_results = sorted(reranked_results, key=lambda x: x["rerank_score"], reverse=True)

    return reranked_results[:k]


# ---------------------------------------------------------------------------
# 3. INTERACTIVE STRESS TESTING BLOCK
# ---------------------------------------------------------------------------
experiment_queries = [
    "airbag requirements",
    "What safety rule protects high-voltage switches from locking up in sub-zero winter environments during a security breach?",
    "system relay latches at negative 30 degrees cold start with hack attempt",
    "Show me a QM rated requirement involving a low exposure rate (E2) despite dealing with a potentially fatal (S3) cold thermal condition"
]

print("\n🔬 --- RUNNING HYBRID + STAGE-2 RERANKING EXPERIMENT ---")

for q_idx, query in enumerate(experiment_queries, 1):
    print(f"\n==================================================")
    print(f"🔍 Test Query #{q_idx}: '{query}'")
    print(f"==================================================")

    # Pull top results using Hybrid Search + Cross-Encoder Reranking
    hits = hybrid_search_and_rerank(
        query=query,
        embedding_model=model,
        index=faiss_index,
        bm25_retriever=bm25_retriever,
        reranker=reranker_model,
        metadata_storage=metadata,
        documents=docs,
        k=10,             # Return top 10 final matches
        pool_size=25     # Retrieve top 25 candidates to feed into the reranker
    )

    print("🏆 Reranked Top Matches:")
    for rank, hit in enumerate(hits, 1):
        m = hit["metadata"]
        print(f"\n  [{rank}] ID: {m['id']} | True Rating: {m['asil']} (Rerank Score: {hit['rerank_score']:.4f} | RRF Score: {hit['rrf_score']:.4f})")
        print(f"      Text: {m['req_text']}")
        print(f"      Reasoning: {m['reasoning'][:120]}...")

⚡ Variables already active in memory! Skipping Google Drive reload to save time.

🔬 --- RUNNING HYBRID + STAGE-2 RERANKING EXPERIMENT ---

🔍 Test Query #1: 'airbag requirements'


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

🏆 Reranked Top Matches:

  [1] ID: REQ-GEN-0593 | True Rating: QM (Rerank Score: 8.0690 | RRF Score: 0.0161)
      Text: The airbag system shall maintain communication integrity over CAN-FD during highway driving at 130 km/h to ensure timely deployment in the event of a collision.
      Reasoning: The requirement addresses the critical nature of airbag deployment (S3) while recognizing that the probability of commun...

  [2] ID: REQ-GEN-0202 | True Rating: ASIL A (Rerank Score: 8.0424 | RRF Score: 0.0122)
      Text: The airbag system shall provide diagnostic coverage for all critical components during the vehicle hand-over from automated to manual mode to ensure timely deployment in case of an accident.
      Reasoning: This requirement addresses S1 by aiming to mitigate light to moderate injuries through effective airbag deployment. The ...

  [3] ID: REQ-GEN-0535 | True Rating: ASIL A (Rerank Score: 8.0087 | RRF Score: 0.0137)
      Text: The airbag deployment system shall ensure a

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

🏆 Reranked Top Matches:

  [1] ID: REQ-GEN-0110 | True Rating: ASIL B (Rerank Score: 5.1511 | RRF Score: 0.0149)
      Text: The Battery Management System (BMS) shall ensure that power relays do not latch during a cold-start at -30آ°C under any fault condition.
      Reasoning: The requirement addresses a scenario where fatalities are likely (S3) due to potential relay latching, which occurs with...

  [2] ID: REQ-GEN-0176 | True Rating: ASIL B (Rerank Score: 5.1084 | RRF Score: 0.0127)
      Text: The sensor fusion system shall detect and mitigate latching of power relays during cold-start conditions at -30آ°C to ensure safe operation of the urban micro-EV.
      Reasoning: The requirement addresses a scenario where fatalities are likely (S3) due to power relay latching, which can occur with ...

  [3] ID: REQ-GEN-0201 | True Rating: ASIL B (Rerank Score: 4.7229 | RRF Score: 0.0164)
      Text: The body electronics control unit shall ensure that power relays do not latch during cold-s

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

🏆 Reranked Top Matches:

  [1] ID: REQ-GEN-0318 | True Rating: QM (Rerank Score: 4.3079 | RRF Score: 0.0120)
      Text: The thermal management system shall ensure that diagnostic coverage for temperature sensors is maintained at a minimum of 95% during urban stop-and-go traffic.
      Reasoning: Given that S3 indicates fatalities are likely if the thermal management system fails, maintaining a high diagnostic cove...

  [2] ID: REQ-GEN-0737 | True Rating: QM (Rerank Score: 3.8485 | RRF Score: 0.0156)
      Text: The thermal management system shall maintain operational status during cold-start at -30آ°C to ensure communication integrity on CAN/Ethernet.
      Reasoning: The requirement addresses a scenario where loss of communication could lead to fatalities (S3). The likelihood of such a...

  [3] ID: REQ-GEN-0683 | True Rating: QM (Rerank Score: 3.3877 | RRF Score: 0.0123)
      Text: The system shall prevent unintended activation of the propulsion system during cold-start at -30آ°C 